In [1]:
import io
import pandas as pd
from collections import defaultdict
from pathlib import Path
from enum import Enum

In [16]:
# Constants
MONTHS = ["jan", "feb"]
MAIN = "dkb_main"
COMMON = "dkb_common"
DATA_FOLDERS = [MAIN, COMMON]
BANK_FOLDER_PATH = "/home/junnwoo/Documents/bank_data"

# Headers
class Header(Enum):
    DATE = "Date"
    NAME = "Name"
    AMOUNT = "Amount"
    REF_NR = "RefNr"

# Categories
class Category(Enum):
    GROCERIES = "Groceries"
    EAT_OUT = "Eating out"
    ELECTRICITY = "Electricity"
    INSURANCE = "Insurance"
    HOME = "Home"

# DKB constants
DKB_COLS = [0, 4, 8, 11] # booking date, receiver, amount, reference number
DKB_SKIPROWS = 4
DKB_DELIMETER = ";"

In [12]:
accounts: dict[str, list[Path]] = defaultdict(list)
for account in DATA_FOLDERS:
    for month in MONTHS:
        if not account in accounts:
            accounts[account] = []
        accounts[account].append(Path(BANK_FOLDER_PATH + "/" + account + "/" + account + "_" + month + ".csv"))
accounts


defaultdict(list,
            {'dkb_main': [PosixPath('/home/junnwoo/Documents/bank_data/dkb_main/dkb_main_jan.csv'),
              PosixPath('/home/junnwoo/Documents/bank_data/dkb_main/dkb_main_feb.csv')],
             'dkb_common': [PosixPath('/home/junnwoo/Documents/bank_data/dkb_common/dkb_common_jan.csv'),
              PosixPath('/home/junnwoo/Documents/bank_data/dkb_common/dkb_common_feb.csv')]})

In [17]:
def load_dataframe_from_paths(paths: list[Path], delimeter: str, skiprows: int, usecols: list[int]):
    data: list[str] = []
    for path in paths:
        with path.open('r') as f:
            data += f.readlines()[skiprows:]
    data_as_string = "".join(data)
    df = pd.read_csv(io.StringIO(data_as_string), delimiter=delimeter, usecols=usecols)
    df.columns = [h for h in Header]
    df[Header.DATE] = pd.to_datetime(df[Header.DATE], dayfirst=True, format="%d.%m.%y")
    df[Header.NAME] = df[Header.NAME].str.strip()
    df[Header.AMOUNT] = df[Header.AMOUNT].str.replace('.', '').str.replace(',', '.')
    df[Header.AMOUNT] = pd.to_numeric(df[Header.AMOUNT], downcast='float', )
    return df

def load_dkb_dataframe_from_paths(paths: list[Path]):
    return load_dataframe_from_paths(paths, skiprows = DKB_SKIPROWS, delimeter = DKB_DELIMETER, usecols = DKB_COLS)
        

In [30]:
def categorize_expense(row):
    name: str = row[Header.NAME]
    if any(l in name.lower() for l in ["rewe", "lidl", "aldi", "edeka"]):
        return Category.GROCERIES.value
    if any(l in name.lower() for l in ["drogerie.markt", "rossman"]):
        return Category.HOME.value
    if any(l in name.lower() for l in ["feather", "allianz"]):
        return Category.INSURANCE.value
    if any(l in name.lower() for l in ["vattenfall"]):
        return Category.ELECTRICITY.value


In [31]:
# Load data from common account
common_df = load_dkb_dataframe_from_paths(accounts[COMMON])
common_df["Category"] = common_df.apply(categorize_expense, axis=1)
common_df

,Header.DATE,Header.NAME,Header.AMOUNT,Header.REF_NR,Category
0,2026-01-30,Lidl.sagt.Danke/Berlin,-8.54,486029700073995,Groceries
1,2026-01-30,Lidl.sagt.Danke/Berlin,-2.43,486029264185199,Groceries
2,2026-01-29,DM.Drogerie.Markt/Berlin,-3.30,486028649930958,Home
3,2026-01-29,Lidl.sagt.Danke/Berlin,-3.40,486028254476978,Groceries
4,2026-01-28,Rossmann.596/Berlin,-12.22,486027619555527,Home
...,...,...,...,...,...
88,2026-02-02,Allianz Versicherungs-AG,-13.59,AB226028-E01534037,Insurance
89,2026-02-02,VATTENFALL EUROPE SALES ...,-84.00,S/836700659407/603113463006,Electricity
90,2026-02-02,Jung7 GmbH und CO KG,-1770.00,NaN,NaN
91,2026-02-02,My Huyen Vu und Jonathan Akesson,2100.00,NaN,NaN


In [32]:
# Load data from main account
main_df = load_dkb_dataframe_from_paths(accounts[MAIN])
main_df["Category"] = main_df.apply(categorize_expense, axis=1)
main_df

,Header.DATE,Header.NAME,Header.AMOUNT,Header.REF_NR,Category
0,2026-01-30,BUDNI.SAGT.DANKE/BERLIN,-2.290000,486029623732014,NaN
1,2026-01-26,CAFE.RESTAURANT.SIDNEY/BERLIN,-58.299999,486024416349425,NaN
2,2026-01-23,Jonathan Akesson,3404.419922,7cf5bbbd380b4e9b86f5975f8b9bb2f8,NaN
3,2026-01-23,PAYPAL..ductran0896/Visa.Direct,-13.000000,306023313113676,NaN
4,2026-01-22,PAY.nl.Goldies.Chefhai/BERLIN,-15.800000,486021708876693,NaN
5,2026-01-22,DM.Drogerie.Markt/Berlin,-109.949997,486021626969696,Home
6,2026-01-19,My Huyen Vu und Jonathan Akesson,-1200.000000,NaN,NaN
7,2026-01-19,Jonathan Aake Eun Woc Akesson,-50.000000,NaN,NaN
8,2026-01-19,Jonathan Aake Eun Woc Akesson,-100.000000,NaN,NaN
9,2026-01-16,"Akesson, Jonathan",722.750000,2808793071,NaN
